# Zero-Inflated Models for Count Data with Excess Zeros

**Topics:** Zero-Inflation, Two-Part Models, Excess Zeros

## Overview

Handle count data with more zeros than expected from Poisson/Negative Binomial.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aurora.models import fit_glm
from aurora.validation.metrics import root_mean_squared_error

sns.set_style('whitegrid')
np.random.seed(42)

## Generate Zero-Inflated Data

In [ ]:
n = 400
x = np.random.uniform(0, 5, n)

# Two processes:
# 1. Binary: Is there any activity? (structural zeros)
prob_active = 1 / (1 + np.exp(-(-1 + 0.3*x)))
is_active = np.random.binomial(1, prob_active)

# 2. Count: If active, how many events?
lambda_count = np.exp(1 + 0.4*x)
count = np.random.poisson(lambda_count)

# Observed = count if active, else 0
y = is_active * count

df = pd.DataFrame({
    'x': x,
    'y': y,
    'is_active': is_active,
    'count': count
})

# Compare observed vs expected zeros
zeros_observed = (y == 0).sum()
zeros_pct = zeros_observed / n * 100

# Expected zeros from Poisson with same mean
mean_nonzero = y[y > 0].mean() if (y > 0).any() else 1
zeros_expected_pct = np.exp(-mean_nonzero) * 100

print(f"Generated {n} observations")
print(f"\nZero counts:")
print(f"  Observed: {zeros_observed} ({zeros_pct:.1f}%)")
print(f"  Expected from Poisson: ~{zeros_expected_pct:.1f}%")
print(f"  Excess zeros: {zeros_pct - zeros_expected_pct:.1f}%")
print(f"\n→ {'Zero-inflated model needed!' if zeros_pct > zeros_expected_pct * 1.5 else 'Regular Poisson OK'}")

## Visualize Zero-Inflation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with excess zeros
axes[0].hist(y, bins=range(0, int(y.max())+2), edgecolor='k', alpha=0.7, align='left')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label=f'Zeros: {zeros_observed}')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Zero-Inflated Counts\n({zeros_pct:.1f}% zeros)')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Scatter with structural zeros highlighted
axes[1].scatter(df[df['is_active']==1]['x'], df[df['is_active']==1]['y'], 
                alpha=0.6, label='Active (count > 0 possible)', edgecolor='k', linewidth=0.5)
axes[1].scatter(df[df['is_active']==0]['x'], df[df['is_active']==0]['y'], 
                alpha=0.6, color='red', label='Inactive (structural zero)', marker='x', s=50)
axes[1].set_xlabel('Predictor X')
axes[1].set_ylabel('Count Y')
axes[1].set_title('Structural vs Sampling Zeros')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Two-Part Model Approach

### Part 1: Binary model for any activity

In [ ]:
X = np.column_stack([np.ones(n), x])

# Binary: any vs none
y_binary = (y > 0).astype(int)

result_binary = fit_glm(X=X, y=y_binary, family='binomial')

print("Part 1: Binary Model (Active vs Inactive)")
print(f"\nIntercept: {result_binary.coef_[0]:.3f}")
print(f"Slope: {result_binary.coef_[1]:.3f}")
print(f"\nInterpretation: Higher X -> higher probability of being active")

### Part 2: Count model for positive observations

In [ ]:
# Filter to positive counts only
mask_positive = y > 0
X_pos = X[mask_positive]
y_pos = y[mask_positive]

result_count = fit_glm(X=X_pos, y=y_pos, family='poisson')

print("\nPart 2: Count Model (Given Active)")
print(f"\nIntercept: {result_count.coef_[0]:.3f}")
print(f"Slope: {result_count.coef_[1]:.3f}")
print(f"\nInterpretation: Among active cases, higher X -> higher count")

## Combined Predictions

In [ ]:
# Predict probability of being active
prob_active_pred = result_binary.predict(X)

# Predict count if active
count_pred = result_count.predict(X)

# Combined prediction
y_pred = prob_active_pred * count_pred

# Compare to naive Poisson
result_naive = fit_glm(X=X, y=y, family='poisson')
y_pred_naive = result_naive.predict(X)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.5, label='Two-part model', s=30)
plt.scatter(y, y_pred_naive, alpha=0.5, label='Naive Poisson', s=30)
plt.plot([0, y.max()], [0, y.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Observed Count')
plt.ylabel('Predicted Count')
plt.title('Zero-Inflated: Two-Part vs Naive Poisson')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# RMSE comparison
from aurora.validation.metrics import root_mean_squared_error
rmse_twopart = root_mean_squared_error(y, y_pred)
rmse_naive = root_mean_squared_error(y, y_pred_naive)

print(f"\nRMSE Comparison:")
print(f"  Two-part model: {rmse_twopart:.3f}")
print(f"  Naive Poisson: {rmse_naive:.3f}")
print(f"\nTwo-part model {'better' if rmse_twopart < rmse_naive else 'similar'} for zero-inflated data")

## When to Use Zero-Inflated Models

- Excess zeros beyond Poisson/NB expectation
- Two distinct processes (participation + intensity)
- Examples: Insurance claims, species counts, purchase frequency

## Alternatives

1. **Hurdle models**: Similar to two-part but different interpretation
2. **Zero-inflated Poisson/NB**: Native implementation (if available)
3. **Mixture models**: Latent class approach

**Next:** `04_longitudinal/01_repeated_measures.ipynb` for within-subject correlation